# Delta Lake: Change Data Feed


## Silver Table

- Create a silver table that tracks absolute number vaccinations and available doses by country

In [0]:
countries = [("USA", 10000, 20000), ("India", 1000, 1500), ("UK", 7000, 10000), ("Canada", 500, 700) ]
columns = ["Country","NumVaccinated","AvailableDoses"]
(spark.createDataFrame(data=countries, schema = columns)
 .write.format("delta")
 .mode("overwrite")
 .saveAsTable("bde2025.lakehouse.silverTable"))

In [0]:
%sql
SELECT * FROM silverTable

Country,NumVaccinated,AvailableDoses
Canada,500,700
India,1000,1500
USA,10000,20000
UK,7000,10000



## Gold Table

- Generate gold table showing vaccination rate by country

In [0]:
import pyspark.sql.functions as F
(spark.read.format("delta")
 .table("bde2025.lakehouse.silverTable")
 .withColumn("VaccinationRate", F.col("NumVaccinated") / F.col("AvailableDoses"))
 .drop("NumVaccinated").drop("AvailableDoses")
 .write.format("delta").mode("overwrite").saveAsTable("bde2025.lakehouse.goldTable"))

In [0]:
%sql
SELECT * FROM goldTable

Country,VaccinationRate
Canada,0.7142857142857143
India,0.6666666666666666
USA,0.5
UK,0.7


## Enable Change Data Feed

In [0]:
%sql
ALTER TABLE silverTable SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

## Update Silver Table Periodically


### Add new country data

In [0]:
# Insert new records
new_countries = [("Australia", 100, 3000)]
(spark.createDataFrame(data=new_countries, schema = columns)
 .write
 .format("delta")
 .mode("append").saveAsTable("bde2025.lakehouse.silverTable"))

### Update existing country data

In [0]:
%sql
-- update a record
UPDATE bde2025.lakehouse.silverTable SET NumVaccinated = '11000' WHERE Country = 'USA'

num_affected_rows
1


In [0]:
%sql
SELECT * FROM bde2025.lakehouse.silverTable

Country,NumVaccinated,AvailableDoses
USA,11000,20000
Australia,100,3000
Canada,500,700
India,1000,1500
UK,7000,10000



## Table History

In [0]:
%sql

DESCRIBE HISTORY bde2025.lakehouse.silverTable

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2023-12-18T09:29:28.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,UPDATE,"Map(predicate -> [""(Country#5112 = USA)""])",null,List(819222920531895),1218-083024-jagx439m,7,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1148, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 3912, scanTimeMs -> 1979, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1365, rewriteTimeMs -> 1902)",null,Databricks-Runtime/12.2.x-scala2.12
7,2023-12-18T09:28:25.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(819222920531895),1218-083024-jagx439m,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1190, numOutputRows -> 1)",null,Databricks-Runtime/12.2.x-scala2.12
6,2023-12-18T09:27:54.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(819222920531895),1218-083024-jagx439m,5,WriteSerializable,true,Map(),null,Databricks-Runtime/12.2.x-scala2.12
5,2023-12-18T09:25:49.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,CREATE OR REPLACE TABLE AS SELECT,"Map(description -> null, isManaged -> true, partitionBy -> [], properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(819222920531895),1218-083024-jagx439m,4,WriteSerializable,false,"Map(numFiles -> 4, numOutputBytes -> 4620, numOutputRows -> 4)",null,Databricks-Runtime/12.2.x-scala2.12
4,2023-07-11T08:04:26.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,UPDATE,"Map(predicate -> [""(Country#4473 = USA)""])",null,List(819222920531895),0711-041046-9vf93uot,3,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1365, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 2532, scanTimeMs -> 894, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1365, rewriteTimeMs -> 1637)",null,Databricks-Runtime/12.2.x-scala2.12
3,2023-07-11T08:02:39.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,UPDATE,"Map(predicate -> [""(Country#3503 = USA)""])",null,List(819222920531895),0711-041046-9vf93uot,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1148, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 4530, scanTimeMs -> 2626, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1365, rewriteTimeMs -> 1872)",null,Databricks-Runtime/12.2.x-scala2.12
2,2023-07-11T08:02:06.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(819222920531895),0711-041046-9vf93uot,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1190, numOutputRows -> 1)",null,Databricks-Runtime/12.2.x-scala2.12
1,2023-07-11T08:01:30.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(819222920531895),0711-041046-9vf93uot,0,WriteSerializable,true,Map(),null,Databricks-Runtime/12.2.x-scala2.12
0,2023-07-11T07:59:53.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,CREATE OR REPLACE TABLE AS SELECT,"Map(description -> null, isManaged -> true, partitionBy -> [], properties -> {})",null,List(819222920531895),0711-041046-9vf93uot,null,WriteSerializable,false,"Map(numFiles -> 4, numOutputBytes -> 4620, numOutputRows -> 4)",null,Databricks-Runtime/12.2.x-scala2.12


In [0]:
%sql
-- update a record
UPDATE bde2025.lakehouse.silverTable SET NumVaccinated = '12000' WHERE Country = 'USA'

num_affected_rows
1


In [0]:
%sql
SELECT * FROM bde2025.lakehouse.silverTable

Country,NumVaccinated,AvailableDoses
USA,12000,20000
Australia,100,3000
Canada,500,700
India,1000,1500
UK,7000,10000


In [0]:
%sql

DESCRIBE HISTORY silverTable

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2023-07-11T08:04:26.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,UPDATE,"Map(predicate -> [""(Country#4473 = USA)""])",null,List(819222920531895),0711-041046-9vf93uot,3,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1365, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 2532, scanTimeMs -> 894, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1365, rewriteTimeMs -> 1637)",null,Databricks-Runtime/12.2.x-scala2.12
3,2023-07-11T08:02:39.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,UPDATE,"Map(predicate -> [""(Country#3503 = USA)""])",null,List(819222920531895),0711-041046-9vf93uot,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1148, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 4530, scanTimeMs -> 2626, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1365, rewriteTimeMs -> 1872)",null,Databricks-Runtime/12.2.x-scala2.12
2,2023-07-11T08:02:06.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(819222920531895),0711-041046-9vf93uot,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1190, numOutputRows -> 1)",null,Databricks-Runtime/12.2.x-scala2.12
1,2023-07-11T08:01:30.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true""})",null,List(819222920531895),0711-041046-9vf93uot,0,WriteSerializable,true,Map(),null,Databricks-Runtime/12.2.x-scala2.12
0,2023-07-11T07:59:53.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,CREATE OR REPLACE TABLE AS SELECT,"Map(description -> null, isManaged -> true, partitionBy -> [], properties -> {})",null,List(819222920531895),0711-041046-9vf93uot,null,WriteSerializable,false,"Map(numFiles -> 4, numOutputBytes -> 4620, numOutputRows -> 4)",null,Databricks-Runtime/12.2.x-scala2.12



## Time Travel

In [0]:
%sql

select * from bde2025.lakehouse.silverTable VERSION AS OF 0

Country,NumVaccinated,AvailableDoses
Canada,500,700
India,1000,1500
USA,10000,20000
UK,7000,10000


In [0]:
%sql

select * from bde2025.lakehouse.silverTable VERSION AS OF 4

Country,NumVaccinated,AvailableDoses
USA,12000,20000
Australia,100,3000
Canada,500,700
India,1000,1500
UK,7000,10000


## Audit Logs

In [0]:
%sql 
-- view the changes
SELECT * FROM table_changes('bde2025.lakehouse.silverTable', 2, 4) order by _commit_timestamp

Country,NumVaccinated,AvailableDoses,_change_type,_commit_version,_commit_timestamp
Australia,100,3000,insert,2,2023-07-11T08:02:06.000+0000
USA,10000,20000,update_preimage,3,2023-07-11T08:02:39.000+0000
USA,11000,20000,update_postimage,3,2023-07-11T08:02:39.000+0000
USA,11000,20000,update_preimage,4,2023-07-11T08:04:26.000+0000
USA,12000,20000,update_postimage,4,2023-07-11T08:04:26.000+0000


In [0]:
changes_df = (spark.read.format("delta")
              .option("readChangeData", True)
              .option("startingVersion", 2)
              .table('bde2025.lakehouse.silverTable'))
display(changes_df)

Country,NumVaccinated,AvailableDoses,_change_type,_commit_version,_commit_timestamp
USA,10000,20000,update_preimage,3,2023-07-11T08:02:39.000+0000
USA,11000,20000,update_postimage,3,2023-07-11T08:02:39.000+0000
USA,11000,20000,update_preimage,4,2023-07-11T08:04:26.000+0000
USA,12000,20000,update_postimage,4,2023-07-11T08:04:26.000+0000
Australia,100,3000,insert,2,2023-07-11T08:02:06.000+0000


## Propagate Changes: Silver to Gold Table

In [0]:
%sql
-- Collect only the latest version for each country
CREATE OR REPLACE TEMPORARY VIEW bde2025.lakehouse.silverTable_latest_version as
SELECT * 
    FROM 
         (SELECT *, rank() over (partition by Country order by _commit_version desc) as rank
          FROM table_changes('bde2025.lakehouse.silverTable', 2, 5)
          WHERE _change_type !='update_preimage')
    WHERE rank=1

In [0]:
%sql

SELECT * FROM bde2025.lakehouse.silverTable_latest_version

Country,NumVaccinated,AvailableDoses,_change_type,_commit_version,_commit_timestamp,rank
Australia,100,3000,insert,2,2023-07-11T08:02:06.000+0000,1
USA,12000,20000,update_postimage,4,2023-07-11T08:04:26.000+0000,1


In [0]:
%sql
-- Merge the changes to gold
MERGE INTO bde2025.lakehouse.goldTable t USING bde2025.lakehouse.silverTable_latest_version s ON s.Country = t.Country
        WHEN MATCHED AND s._change_type='update_postimage' THEN UPDATE SET VaccinationRate = s.NumVaccinated/s.AvailableDoses
        WHEN NOT MATCHED THEN INSERT (Country, VaccinationRate) VALUES (s.Country, s.NumVaccinated/s.AvailableDoses)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM bde2025.lakehouse.goldTable

Country,VaccinationRate
Australia,0.03333333333333333
USA,0.6
Canada,0.7142857142857143
India,0.6666666666666666
UK,0.7



## History and Time Travel of Gold Table

In [0]:
%sql 

DESCRIBE HISTORY bde2025.lakehouse.goldTable

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2023-07-11T08:11:17.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,MERGE,"Map(matchedPredicates -> [{""predicate"":""(_change_type#5976 = update_postimage)"",""actionType"":""update""}], notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], predicate -> [""(Country#5973 = Country#5999)""])",null,List(819222920531895),0711-041046-9vf93uot,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 900, numTargetBytesRemoved -> 877, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 9183, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, scanTimeMs -> 4388, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2304)",null,Databricks-Runtime/12.2.x-scala2.12
0,2023-07-11T08:00:55.000+0000,6372822039691303,manaranjan.pradhan@iimb.ac.in,CREATE OR REPLACE TABLE AS SELECT,"Map(description -> null, isManaged -> true, partitionBy -> [], properties -> {})",null,List(819222920531895),0711-041046-9vf93uot,null,WriteSerializable,false,"Map(numFiles -> 4, numOutputBytes -> 3536, numOutputRows -> 4)",null,Databricks-Runtime/12.2.x-scala2.12


In [0]:
%sql 

SELECT * FROM bde2025.lakehouse.goldTable VERSION AS OF 0

Country,VaccinationRate
Canada,0.7142857142857143
India,0.6666666666666666
USA,0.5
UK,0.7


In [0]:
%sql 

SELECT * FROM bde2025.lakehouse.goldTable VERSION AS OF 1

Country,VaccinationRate
Australia,0.03333333333333333
USA,0.6
Canada,0.7142857142857143
India,0.6666666666666666
UK,0.7



## Compute change across multiple versions

In [0]:
gold_v0 = spark.sql('SELECT country, VaccinationRate as vc_v0 FROM bde2025.lakehouse.goldTable VERSION AS OF 0')
gold_v1 = spark.sql('SELECT country, VaccinationRate as vc_v1 FROM bde2025.lakehouse.goldTable VERSION AS OF 1')

In [0]:
from pyspark.sql.functions import col, format_number

In [0]:
gold_changes = (gold_v0.join(gold_v1, on = ['Country'], how = 'inner')
                .select('country', 
                        format_number(col('vc_v0'), 2).alias('vc_v0'), 
                        format_number(col('vc_v1'), 2).alias('vc_v1'))
                .withColumn('vc_change', format_number((col('vc_v1') - col('vc_v0')) / col('vc_v0'), 2) ))                

In [0]:
display(gold_changes)

country,vc_v0,vc_v1,vc_change
USA,0.50,0.60,0.20
Canada,0.71,0.71,0.00
India,0.67,0.67,0.00
UK,0.70,0.70,0.00


## Clean up tables

In [0]:
%sql
DROP TABLE bde2025.lakehouse.silverTable;
DROP TABLE bde2025.lakehouse.goldTable;